# Day 6: Pressure Prediction Baseline

Extends the Day 5 MLP (Cl + Cd) to predict the full pressure coefficient distribution (Cp) alongside Cl and Cd.

**Inputs:** Airfoil Geometry (198) + AoA + Reynolds + Ncrit = 201  
**Outputs:** Cl + Cd + Cp (98) = 100  

Day 5 artifacts (`baseline_mlp.pt`, `baseline_metrics.json`) are preserved and not overwritten.

In [11]:
# Setup: download and preprocess data (skip if already done from Day 5)
import os

if not os.path.exists("processed_output/standard/main/train.npz"):
    os.system("wget https://nasa-public-data.s3.amazonaws.com/plot3d_utilities/dataset-processed.zip")
    os.system("unzip dataset-processed.zip")
    os.system("python preprocess_airfoil_dataset.py --input_dir datasets/standard --output_dir processed_output --scaler_name standard")
else:
    print("Processed data already exists, skipping download and preprocessing.")

Processed data already exists, skipping download and preprocessing.


In [12]:
import numpy as np

train = np.load("processed_output/standard/main/train.npz")
val   = np.load("processed_output/standard/main/val.npz")
test  = np.load("processed_output/standard/main/test.npz")

print("Train samples:", train["cl"].shape[0])
print("Val   samples:", val["cl"].shape[0])
print("Test  samples:", test["cl"].shape[0])

Train samples: 457283
Val   samples: 80696
Test  samples: 230563


In [13]:
# Build feature matrices — same as Day 5
def make_X(split):
    return np.hstack([
        split["geometry_y"],
        split["alpha"].reshape(-1, 1),
        split["reynolds"].reshape(-1, 1),
        split["ncrit"].reshape(-1, 1)
    ])

# Updated target: Cl + Cd + Cp (100 outputs)
def make_y(split):
    return np.concatenate([
        split["cl"].reshape(-1, 1),
        split["cd"].reshape(-1, 1),
        split["cp"]
    ], axis=1)

X_train = make_X(train);  y_train = make_y(train)
X_val   = make_X(val);    y_val   = make_y(val)
X_test  = make_X(test);   y_test  = make_y(test)

print("X_train:", X_train.shape)  # (N, 201)
print("y_train:", y_train.shape)  # (N, 100)

X_train: (457283, 201)
y_train: (457283, 100)


In [14]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def to_loader(X, y, batch_size=1024, shuffle=True):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_train, y_train, shuffle=True)
val_loader   = to_loader(X_val,   y_val,   shuffle=False)

In [15]:
import torch.nn as nn

class MLP(nn.Module):
    """
    Same architecture as Day 5 except output layer is 100 (Cl + Cd + Cp[98])
    instead of 2 (Cl + Cd only).
    """
    def __init__(self, out_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(201, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, out_dim)
        )

    def forward(self, x):
        return self.net(x)

model = MLP(out_dim=100).cuda()
print(model)

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
best_val_loss = float("inf")

for epoch in range(20):
    # Train
    model.train()
    running_loss = 0.0
    for x, y in train_loader:
        x, y = x.cuda(), y.cuda()
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.cuda(), y.cuda()
            val_loss += criterion(model(x), y).item()
    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1:02d} | Train Loss = {train_loss:.6f} | Val Loss = {val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "baseline_cl_cd_cp.pt")

print("\nTraining complete. Best Val Loss:", best_val_loss)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import json

# Load best checkpoint
model.load_state_dict(torch.load("baseline_cl_cd_cp.pt"))
model.eval()

X_val_t = torch.tensor(X_val, dtype=torch.float32).cuda()
with torch.no_grad():
    pred = model(X_val_t).cpu().numpy()

# Slice outputs
pred_cl = pred[:, 0];   true_cl = y_val[:, 0]
pred_cd = pred[:, 1];   true_cd = y_val[:, 1]
pred_cp = pred[:, 2:];  true_cp = y_val[:, 2:]

# Overall
overall_mae  = mean_absolute_error(y_val, pred)
overall_rmse = np.sqrt(mean_squared_error(y_val, pred))
overall_r2   = r2_score(y_val, pred)

# Per-target
cl_mae  = mean_absolute_error(true_cl, pred_cl)
cl_rmse = np.sqrt(mean_squared_error(true_cl, pred_cl))
cd_mae  = mean_absolute_error(true_cd, pred_cd)
cd_rmse = np.sqrt(mean_squared_error(true_cd, pred_cd))
cp_mae  = mean_absolute_error(true_cp, pred_cp)
cp_rmse = np.sqrt(mean_squared_error(true_cp, pred_cp))

print(f"Overall  | MAE={overall_mae:.6f}  RMSE={overall_rmse:.6f}  R²={overall_r2:.6f}")
print(f"Cl       | MAE={cl_mae:.6f}  RMSE={cl_rmse:.6f}")
print(f"Cd       | MAE={cd_mae:.6f}  RMSE={cd_rmse:.6f}")
print(f"Cp       | MAE={cp_mae:.6f}  RMSE={cp_rmse:.6f}")

metrics = {
    "overall_mae": overall_mae,
    "overall_rmse": overall_rmse,
    "overall_r2": overall_r2,
    "cl_mae": cl_mae,
    "cl_rmse": cl_rmse,
    "cd_mae": cd_mae,
    "cd_rmse": cd_rmse,
    "cp_mae": cp_mae,
    "cp_rmse": cp_rmse
}

with open("baseline_cl_cd_cp_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\nMetrics saved to baseline_cl_cd_cp_metrics.json")

In [ ]:
import matplotlib.pyplot as plt

# Plot Cp ground truth vs prediction for 3 validation samples
sample_indices = [0, 500, 1000]
x_surface = np.linspace(0, 1, 98)  # normalized chord position

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, idx in enumerate(sample_indices):
    ax = axes[i]
    ax.plot(x_surface, true_cp[idx], label="Ground Truth", color="steelblue", linewidth=2)
    ax.plot(x_surface, pred_cp[idx], label="Predicted",    color="tomato",    linewidth=2, linestyle="--")
    ax.invert_yaxis()  # Cp convention: lower values plotted upward
    ax.set_xlabel("Chord Position (x/c)")
    ax.set_ylabel("Cp")
    ax.set_title(f"Validation Sample {idx}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Pressure Coefficient Distribution: Ground Truth vs Predicted", fontsize=13)
plt.tight_layout()
plt.savefig("cp_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: cp_comparison.png")

In [ ]:
# Confirm Day 5 artifacts are untouched
import os
for f in ["baseline_mlp.pt", "baseline_metrics.json"]:
    exists = os.path.exists(f"artifacts/{f}") or os.path.exists(f)
    print(f"{f}: {'EXISTS' if exists else 'MISSING — copy from Day 5 before committing'}")

print("\nDay 6 artifacts:")
for f in ["baseline_cl_cd_cp.pt", "baseline_cl_cd_cp_metrics.json", "cp_comparison.png"]:
    print(f"  {f}: {'EXISTS' if os.path.exists(f) else 'MISSING'}")